# Fig. 1h — Combined VHC + VGN matrix map (DEG + SCENIC GRN)

Two stacked matrix plots over the same row order (9 VHC + 13 VGN clusters):
1. **DEG matrix** — top differentially expressed genes per cluster, block-ordered, with selected markers (red in final figure) overlaid.
2. **SCENIC GRN matrix** — top regulons per cluster from pySCENIC, summarized as cluster-mean raw AUC → z-score → row-wise 0–1.

**Inputs**
- `Vestibular_VGN_VHC_merged.h5ad` — reference adata (used for the cluster colour palette and shared categorical order)
- `Vestibular_VHC_filtered.h5ad`, `Vestibular_VGN_filtered.h5ad` — contamination-filtered VHC / VGN adata (used for DEG)
- `SCENIC_VHC.loom`, `SCENIC_VGN.loom` — pySCENIC outputs (RegulonsAUC + regulon metadata)

**Outputs**
- `figures/Fig1h_DEG_matrix.pdf` — DEG block matrix with curved gene labels
- `figures/Fig1h_SCENIC_matrix.pdf` — top-5 staircase regulon matrix
- `figures/Fig1h_DEG_values.csv`, `figures/Fig1h_SCENIC_values.csv` — underlying matrices

**Upstream**
- Contamination-filtered adata are produced upstream (4-axis blacklist on stress / mito-ribo / glia / metabolic + covariate ρ filter).
- SCENIC loom files are produced by the standard pySCENIC pipeline: GRNBoost2 → cisTarget → AUCell.
- This notebook only handles the final matrix plotting.

In [ ]:
import json
import zlib
import base64
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import loompy as lp
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from scipy import sparse
from scipy.sparse import issparse

from PyComplexHeatmap import ClusterMapPlotter, HeatmapAnnotation
from PyComplexHeatmap.annotations import anno_label

sc.settings.verbosity = 1
sc.set_figure_params(figsize=(6, 5), dpi_save=300, dpi=100, frameon=False)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['pcolor.shading'] = 'auto'

DATA_DIR = Path('..')
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)

## 1. Shared cluster order

In [ ]:
HC_ORDER = [
    'HC_Gsn', 'HC_Kcnv1', 'HC_Socs3', 'HC_St3gal6', 'HC_Atp2a3',
    'HC_Tshz3', 'HC_Foxp2', 'HC_Agbl1', 'HC_Car8',
]
NEURON_ORDER = [
    'neuron_Gstm2', 'neuron_Fxyd7', 'neuron_Slc2a4', 'neuron_Lmo3', 'neuron_Meis2',
    'neuron_Vglut1/Aldh1a3', 'neuron_Vglut1/Lypd1', 'neuron_Vglut1/Calb1',
    'neuron_Vglut1/Nkain3', 'neuron_Vglut1/Calb2',
    'neuron_Nxph4', 'neuron_Gsg1l', 'neuron_Ptgfr',
]
ROW_ORDER = HC_ORDER + NEURON_ORDER

# Part 1 — DEG matrix

Steps:
1. Load contamination-filtered VHC + VGN adata.
2. `rank_genes_groups` per dataset (Welch t-test on `log_norm` layer).
3. For each cluster, take top `N_SHOW` genes; allow duplicates across cluster blocks (a gene strongly expressed in two clusters appears twice).
4. Compute cluster-mean expression on the `umi` layer, then min-max normalise within HC vs neuron blocks so the two cell types share comparable colour scaling.
5. Plot with `PyComplexHeatmap.anno_label` for curved gene labels.

In [ ]:
adata_all_ref = sc.read_h5ad(DATA_DIR / 'Vestibular_VGN_VHC_merged.h5ad')
adata_HC = sc.read_h5ad(DATA_DIR / 'Vestibular_VHC_filtered.h5ad')
adata_neuron = sc.read_h5ad(DATA_DIR / 'Vestibular_VGN_filtered.h5ad')

# fix categorical order so rank_genes_groups columns come out sorted
for subset_adata, subset_order in [(adata_HC, HC_ORDER), (adata_neuron, NEURON_ORDER)]:
    present = [c for c in subset_order if c in subset_adata.obs['all_cluster_gene_name'].astype(str).unique()]
    subset_adata.obs['all_cluster_gene_name'] = pd.Categorical(
        subset_adata.obs['all_cluster_gene_name'].astype(str),
        categories=present, ordered=True,
    )

print('VHC :', adata_HC.shape, '| VGN :', adata_neuron.shape)

In [ ]:
# union adata for cluster-mean expression on the same gene universe
shared_genes = sorted(set(adata_HC.var_names) & set(adata_neuron.var_names))
adata_all_0407 = ad.concat(
    [adata_HC[:, shared_genes], adata_neuron[:, shared_genes]],
    axis=0, label='dataset', keys=['HC', 'neuron'], join='outer',
)
adata_all_0407.obs['all_cluster_gene_name'] = pd.Categorical(
    adata_all_0407.obs['all_cluster_gene_name'].astype(str),
    categories=ROW_ORDER, ordered=True,
)
print('Combined adata:', adata_all_0407.shape)

In [ ]:
# DEG per cluster, on log_norm layer (Welch t-test overestim var, same as in the paper)
for ad_obj in (adata_HC, adata_neuron):
    sc.tl.rank_genes_groups(
        ad_obj, groupby='all_cluster_gene_name',
        method='t-test_overestim_var', n_genes=50,
        layer='log_norm', use_raw=False,
    )

In [ ]:
def group_means(adata, genes, groupby, layer='umi'):
    X = adata.layers[layer]
    if sparse.issparse(X):
        X = X.toarray()
    gene_idx = adata.var_names.get_indexer(genes)
    if (gene_idx < 0).any():
        missing = [genes[i] for i in np.where(gene_idx < 0)[0]]
        raise ValueError(f'Genes not in var_names: {missing[:10]}')
    df = pd.DataFrame(X[:, gene_idx], columns=genes)
    df[groupby] = adata.obs[groupby].astype(str).values
    return df.groupby(groupby, observed=True).mean()

def blockwise_minmax(df, blocks):
    out = df.copy()
    for _, rows in blocks.items():
        sub = out.loc[rows]
        denom = (sub.max(axis=0) - sub.min(axis=0)).replace(0, np.nan)
        out.loc[rows] = (sub - sub.min(axis=0)) / denom
    return out.fillna(0.0)

def top_n_per_cluster(adata, n=20, extras=None):
    extras = extras or {}
    rg = adata.uns['rank_genes_groups']['names']
    return {
        str(g): list(dict.fromkeys(list(rg[g][:n]) + extras.get(str(g), [])))
        for g in rg.dtype.names
    }

In [ ]:
N_SHOW = 20
LABEL_TOP_N = 3

# Curated markers to highlight in red (force into the layout near their dominant cluster)
manual_genes = [
    'Kcnv1', 'Foxp2', 'Atp2a3', 'Agbl1', 'Gsn', 'St3gal6', 'Tshz3',
    'Lmo3', 'Gsg1l', 'Meis2', 'Lypd1', 'Nkain3', 'Slc2a4', 'Ptgfr',
    'Calb1', 'Calb2',
]
manual_owner_seed = {
    'Kcnv1': 'HC_Kcnv1', 'Foxp2': 'HC_Foxp2', 'Atp2a3': 'HC_Atp2a3',
    'Agbl1': 'HC_Agbl1', 'Gsn': 'HC_Gsn', 'St3gal6': 'HC_St3gal6', 'Tshz3': 'HC_Tshz3',
    'Lmo3': 'neuron_Lmo3', 'Gsg1l': 'neuron_Gsg1l', 'Meis2': 'neuron_Meis2',
    'Lypd1': 'neuron_Vglut1/Lypd1', 'Nkain3': 'neuron_Vglut1/Nkain3',
    'Slc2a4': 'neuron_Slc2a4', 'Ptgfr': 'neuron_Ptgfr',
    'Calb1': 'neuron_Vglut1/Calb1', 'Calb2': 'neuron_Vglut1/Calb2',
}

top_hc = top_n_per_cluster(adata_HC, n=N_SHOW)
top_neuron = top_n_per_cluster(adata_neuron, n=N_SHOW)

rows_hc = [r for r in HC_ORDER if r in adata_all_0407.obs['all_cluster_gene_name'].astype(str).unique()]
rows_neuron = [r for r in NEURON_ORDER if r in adata_all_0407.obs['all_cluster_gene_name'].astype(str).unique()]
rows_all = rows_hc + rows_neuron

# decide where each manual gene lives (use seed mapping if present, else argmax cluster mean)
manual_present = [g for g in manual_genes if g in adata_all_0407.var_names]
manual_means = group_means(adata_all_0407, manual_present, groupby='all_cluster_gene_name', layer='umi').loc[rows_all, manual_present]
manual_owner = {
    g: (manual_owner_seed.get(g) if manual_owner_seed.get(g) in rows_all else manual_means[g].idxmax())
    for g in manual_present
}

# build per-cluster blocks: top-N DEG + manual genes whose owner is this cluster
cluster_blocks = []
for cl in rows_all:
    src = top_hc if cl in HC_ORDER else top_neuron
    block = list(src.get(cl, []))
    for g in manual_present:
        if manual_owner[g] == cl and g not in block:
            block.append(g)
    cluster_blocks.append((cl, block))

In [ ]:
# expand blocks to columns; duplicate genes get distinct internal IDs
column_records = []
for cl, block in cluster_blocks:
    for rank, gene in enumerate(block, start=1):
        if gene not in adata_all_0407.var_names:
            continue
        column_records.append({
            'block': cl, 'rank_in_block': rank, 'gene': gene,
            'is_manual': gene in manual_genes,
        })
column_meta_df = pd.DataFrame(column_records).reset_index(drop=True)
column_meta_df['internal_id'] = [f'{r.block}__{r.rank_in_block:02d}__{r.gene}' for r in column_meta_df.itertuples()]
column_meta_df['plot_pos'] = np.arange(len(column_meta_df))
column_meta_df['label_me'] = (column_meta_df['rank_in_block'] <= LABEL_TOP_N) | column_meta_df['is_manual']

# expand cluster-mean matrix into duplicated columns
unique_genes = column_meta_df['gene'].drop_duplicates().tolist()
M_unique = group_means(adata_all_0407, unique_genes, groupby='all_cluster_gene_name', layer='umi').loc[rows_all]
M_block = pd.DataFrame(
    {r.internal_id: M_unique[r.gene] for r in column_meta_df.itertuples()},
    index=rows_all,
)
M_block_vis = blockwise_minmax(M_block, blocks={'HC': rows_hc, 'neuron': rows_neuron})

block_sizes = column_meta_df.groupby('block', sort=False).size().reindex(rows_all).fillna(0).astype(int)
M_block_vis.to_csv(FIG_DIR / 'Fig1h_DEG_values.csv')
print('DEG matrix:', M_block_vis.shape)

In [ ]:
# plot via PyComplexHeatmap so curved labels stay aligned with columns
label_values = [g if flag else np.nan for g, flag in zip(column_meta_df['gene'], column_meta_df['label_me'])]
label_df = pd.DataFrame({'gene_label': label_values}, index=M_block_vis.columns)
# manual markers labelled in red; top-N DEG in black
label_colors = {
    g: ('red' if g in manual_genes else 'black')
    for g in column_meta_df.loc[column_meta_df['label_me'], 'gene'].unique()
}

cmap_wdg = mcolors.LinearSegmentedColormap.from_list(
    'white_darkgrey', cm.Greys(np.linspace(0.0, 0.7, 256))
)

plt.close('all')
plt.figure(figsize=(12, 9.5))

bottom_ha = HeatmapAnnotation(
    gene_label=anno_label(
        label_df, colors=label_colors, merge=False, extend=True,
        frac=0.25, rad=0.12, adjust_color=False, legend=False,
        rotation=90, fontsize=7, annotation_clip=False,
        arrowprops=dict(linewidth=0.8),
    ),
    axis=1, verbose=0, legend=False, label_kws=dict(visible=False),
)

cm_pch = ClusterMapPlotter(
    data=M_block_vis,
    row_cluster=False, col_cluster=False,
    row_dendrogram=False, col_dendrogram=False,
    show_rownames=True, show_colnames=False,
    bottom_annotation=bottom_ha,
    cmap=cmap_wdg, vmin=0, vmax=1,
    yticklabels_kws={'labelsize': 8}, legend=False,
    plot=True, verbose=0,
)
cm_pch.ax_heatmap.set_title('Fig 1h — DEG matrix (VHC + VGN, block-ordered)', fontsize=11)

boundary = 0
for cl, size in block_sizes.items():
    boundary += int(size)
    cm_pch.ax_heatmap.axvline(boundary, color='lightgrey', linewidth=1.0)
    if getattr(cm_pch, 'ax_bottom_annotation', None) is not None:
        cm_pch.ax_bottom_annotation.axvline(boundary, color='lightgrey', linewidth=1.0)

plt.savefig(FIG_DIR / 'Fig1h_DEG_matrix.pdf', dpi=300, bbox_inches='tight')
plt.show()

# Part 2 — SCENIC GRN matrix

Steps:
1. Load `RegulonsAUC` matrix and regulon names from each pySCENIC loom.
2. Align AUC rows to contamination-filtered adata cells.
3. Per dataset: cluster-mean AUC → per-regulon z-score (clip at ±2.5) → row-wise 0–1.
4. Pick top-5 regulons per cluster (excluding a hand-picked stress / non-relevant TF blacklist).
5. Concatenate HC + neuron, keep one column per regulon, order columns to follow row staircase.

In [ ]:
TOP_N = 5
CLIP_Z = 2.5
# regulons removed from top-K picking (stress / contamination-associated TFs)
EXCLUDE_TF = {
    'Spi1(+)', 'Foxo1(+)', 'Nkx6-2(+)', 'Myrf(+)',
    'Irf8(+)', 'Xbp1(+)', 'Creb3(+)', 'Junb(+)', 'Crem(+)',
    'Bhlhe40(+)', 'Irf3(+)', 'Stat3(+)',
}

def strip_regulon_name(name):
    return name.replace('(+)', '').replace('(-)', '').strip()

def load_scenic_auc(loom_path, clean_h5ad_path, label):
    lf = lp.connect(str(loom_path), mode='r', validate=False)
    auc_raw = lf.ca['RegulonsAUC']
    cell_ids = lf.ca['CellID']
    meta = json.loads(zlib.decompress(base64.b64decode(lf.attrs['MetaData'])))
    regulon_names = [x['regulon'] for x in meta['regulonThresholds']]
    lf.close()

    auc_mtx = pd.DataFrame(auc_raw, index=cell_ids, columns=regulon_names)
    adata_clean = sc.read_h5ad(clean_h5ad_path)
    common = [c for c in auc_mtx.index if c in adata_clean.obs_names]

    adata_auc = ad.AnnData(
        X=auc_mtx.loc[common].values,
        obs=adata_clean[common].obs.copy(),
        var=pd.DataFrame(index=auc_mtx.columns),
    )
    adata_auc.layers['auc_init'] = adata_auc.X.copy()
    print(f'[{label}] AUC: {auc_mtx.shape}, matched cells: {len(common)}')
    return adata_auc

In [ ]:
adata_HC_auc = load_scenic_auc(
    DATA_DIR / 'SCENIC_VHC.loom',
    DATA_DIR / 'Vestibular_VHC_filtered.h5ad', 'HC',
)
adata_neuron_auc = load_scenic_auc(
    DATA_DIR / 'SCENIC_VGN.loom',
    DATA_DIR / 'Vestibular_VGN_filtered.h5ad', 'neuron',
)

In [ ]:
def summarize_regulons(adata, row_order, layer='auc_init', top_n=5, clip_z=2.5, exclude=None):
    exclude = set(exclude) if exclude else set()
    X = adata.layers[layer]
    X = X.toarray() if issparse(X) else X.copy()
    groups = adata.obs['all_cluster_gene_name'].astype(str).values
    present = [g for g in row_order if g in np.unique(groups)]

    M = np.vstack([X[groups == g].mean(axis=0) for g in present])
    mu, sd = M.mean(axis=0), M.std(axis=0)
    sd[sd < 1e-10] = 1.0
    Z = np.clip((M - mu) / sd, -clip_z, clip_z)
    row_min, row_max = Z.min(axis=1, keepdims=True), Z.max(axis=1, keepdims=True)
    span = np.where((row_max - row_min) < 1e-10, 1.0, (row_max - row_min))
    Z01 = np.clip((Z - row_min) / span, 0, 1)
    z01_df = pd.DataFrame(Z01, index=present, columns=list(adata.var_names))

    top_per_cluster = {}
    top_set = []
    for i, cl in enumerate(present):
        picked = []
        for idx in np.argsort(Z[i, :])[::-1]:
            reg = adata.var_names[idx]
            if reg in exclude:
                continue
            picked.append(reg)
            if len(picked) == top_n:
                break
        top_per_cluster[cl] = picked
        top_set.extend(picked)
    return {
        'z01': z01_df,
        'top_per_cluster': top_per_cluster,
        'top_regulons': list(dict.fromkeys(top_set)),
    }

hc_summary = summarize_regulons(adata_HC_auc, HC_ORDER, top_n=TOP_N, clip_z=CLIP_Z, exclude=EXCLUDE_TF)
neuron_summary = summarize_regulons(adata_neuron_auc, NEURON_ORDER, top_n=TOP_N, clip_z=CLIP_Z, exclude=EXCLUDE_TF)

In [ ]:
# combine HC + neuron; keep one column per regulon, ordered by first-appearance in the row staircase
genes_union = neuron_summary['top_regulons'] + [
    g for g in hc_summary['top_regulons'] if g not in set(neuron_summary['top_regulons'])
]
df_hc = hc_summary['z01'].reindex(columns=genes_union, fill_value=0.0)
df_neu = neuron_summary['z01'].reindex(columns=genes_union, fill_value=0.0)
df_all = pd.concat([df_hc, df_neu], axis=0)

rows_valid = [o for o in ROW_ORDER if o in df_all.index]
df_all = df_all.loc[rows_valid]

# staircase column order: walk each row, append its top regulons not yet placed
staircase_genes, seen = [], set()
for cl in rows_valid:
    src = hc_summary if cl in HC_ORDER else neuron_summary
    for reg in src['top_per_cluster'].get(cl, []):
        if reg in EXCLUDE_TF or reg in seen:
            continue
        staircase_genes.append(reg)
        seen.add(reg)
df_all = df_all[staircase_genes]
df_all.columns = [strip_regulon_name(g) for g in df_all.columns]
df_all.to_csv(FIG_DIR / 'Fig1h_SCENIC_values.csv')
print('SCENIC matrix:', df_all.shape)

In [ ]:
n_rows, n_cols = df_all.shape
fig = plt.figure(figsize=(max(12, n_cols * 0.35), 9.5))
gs = gridspec.GridSpec(2, 2, height_ratios=[8, 1.5], width_ratios=[30, 1], hspace=0.01, wspace=0.4)
ax, ax_cbar, ax_anno = (fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, 0]))

im = ax.imshow(df_all.values, aspect='auto', vmin=0.5, vmax=1, cmap='cividis')
ax.set_yticks(range(n_rows))
ax.set_yticklabels(df_all.index.tolist(), fontsize=9)
ax.set_xticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(axis='both', which='both', length=0)
ax.set_title(f'Fig 1h — SCENIC regulon matrix (top {TOP_N}/cluster)', fontsize=11)
fig.colorbar(im, cax=ax_cbar)

ax_anno.set_xlim(-0.5, n_cols - 0.5)
ax_anno.set_ylim(0, 1)
ax_anno.axis('off')
for i, gene in enumerate(df_all.columns):
    ax_anno.plot([i, i], [1.0, 0.55], color='black', linewidth=0.8)
    ax_anno.text(i, 0.5, gene, rotation=90, ha='center', va='top', fontsize=7)

plt.savefig(FIG_DIR / 'Fig1h_SCENIC_matrix.pdf', dpi=300, bbox_inches='tight')
plt.show()